# WS 11.2: Beyond Accuracy

In WS 10.2, you trained decision trees on alien diseases and measured accuracy — the fraction of patients the model got right. You also built a confusion matrix to see *which* mistakes the model made.

Today we'll work with real data and discover why accuracy alone can be misleading. Along the way, you'll learn two new measures — **precision** and **recall** — that describe different kinds of model quality.

> **I will not use AI tools on this worksheet.**
>
> **Name:**

### Setup

Run the cell below to load the libraries we'll use today. You've seen `pandas`, `numpy`, and `matplotlib` before. The two new imports — `DecisionTreeClassifier` and `plot_tree` — come from **scikit-learn** (`sklearn`), the machine learning library you used in WS 10.2.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree

---

## Part 1: A Classifier That Does Nothing

We'll start with a famous dataset: the passenger list from the *Titanic* ([source: Kaggle](https://www.kaggle.com/c/titanic)). Each row is one passenger, and the column `Survived` is 1 if the passenger survived and 0 if they did not.

Run the cell below to load the data.

In [ ]:
titanic = pd.read_csv("https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv")
titanic.head(10)

**Exercise 1.1:** How many passengers are in the dataset? How many survived? How many did not?

*Hint:* `titanic["Survived"].sum()` counts the survivors (since survived = 1). Use `len(titanic)` for the total.

In [ ]:
# Your code here

*Your answer here.*

**Exercise 1.2:** What fraction of passengers survived? What fraction did not?

*Your answer here.*

Imagine the simplest possible "model" — one that does not look at any information about the passenger at all. It just predicts the **most common outcome** for everyone.

**Exercise 1.3:** Based on your answer to Exercise 1.2, what is the most common outcome — survived or did not survive? If we predicted that outcome for every single passenger, what would the accuracy be?

*Your answer here.*

### Creating an array of zeros with `np.zeros()`

To build this model, we need an array filled with zeros — one zero for each passenger. NumPy has a function for exactly this: `np.zeros()`.

```python
# np.zeros(n) creates an array of n zeros
my_array = np.zeros(5)
print(my_array)   # [0. 0. 0. 0. 0.]
```

**Try it:** Create an array of 8 zeros using `np.zeros()` and print it.

In [ ]:
# Your code here

Now let's use `np.zeros()` to create predictions for every passenger in the Titanic dataset.

In [ ]:
predictions_always_no = np.zeros(len(titanic))
print(predictions_always_no)

**Exercise 1.4:** Compute the accuracy of this "model." In WS 10.2, you computed accuracy by comparing predictions to the true labels. Here's the pattern as a reminder:

```python
is_correct = (predictions == y)
accuracy = is_correct.sum() / len(y)
```

Adapt this for the Titanic data — use `predictions_always_no` and `titanic["Survived"]`. Does the number match what you predicted in Exercise 1.3?

In [ ]:
# Your code here

*Your answer here.*

Now let's look at the confusion matrix.

**Exercise 1.5:** Build the confusion matrix using `pd.crosstab()`, just like in WS 10.2 Part 3. Here's the pattern as a reminder:

```python
pd.crosstab(actual_labels, predictions, rownames=['actual'], colnames=['predicted'])
```

Use `titanic["Survived"]` as the actual labels and `predictions_always_no` as the predictions.

In [ ]:
# Your code here

> **Reminder — reading the confusion matrix:**
>
> |  | Predicted **survived** | Predicted **did not survive** |
> |---|---|---|
> | **Actually survived** | True Positive (TP) | False Negative (FN) |
> | **Actually did not survive** | False Positive (FP) | True Negative (TN) |

**Exercise 1.6:** Write the number from each cell:

TP = ___  &nbsp;&nbsp;  FP = ___  &nbsp;&nbsp;  FN = ___  &nbsp;&nbsp;  TN = ___

How many survivors did this model catch? What does this confusion matrix tell you about a model that "just guesses the most common answer"?

*Your answer here.*

This is the key problem: a model that does nothing useful can still have decent accuracy — as long as one outcome is more common than the other. We need better ways to measure quality.

---

## Part 2: Building Real Classifiers

Let's build actual decision trees and see if they do better. We'll use three features that were available on the Titanic: passenger class, sex, and age.

Run the setup cell below. It prepares the features in three steps:

1. **Select columns:** We only keep the four columns we need — `Pclass`, `Sex`, `Age`, and `Survived`. This is the same list-indexing pattern you used in WS 10.2 to select features from a DataFrame.
2. **Drop missing values:** Some passengers have no recorded age. `.dropna()` removes those rows so the tree has complete data to work with. (In a real project, you would need to decide how to handle missing data — for now, dropping is the simplest option.)
3. **Convert text to numbers:** The model needs numbers, not words. `(titanic_clean["Sex"] == "female")` gives `True`/`False`, and `.astype(int)` converts that to 1/0. Behind the scenes, this is the same pattern used in WS 10.2 to create `eyes_orange` and `hair_pink`.

In [ ]:
titanic_clean = titanic[["Pclass", "Sex", "Age", "Survived"]].dropna()
titanic_clean["Sex_female"] = (titanic_clean["Sex"] == "female").astype(int)

features = ["Pclass", "Sex_female", "Age"]
X = titanic_clean[features]
y = titanic_clean["Survived"]
print(len(titanic_clean), "passengers with complete data")

### The always-predict-zero baseline on the clean data

We dropped passengers with missing ages, so the dataset is a bit smaller now. Let's quickly recompute our "predict nobody survived" baseline on the clean data.

In [ ]:
baseline_clean = np.zeros(len(titanic_clean))
accuracy_clean = (baseline_clean == y).sum() / len(y)
print("Baseline accuracy on clean data:", round(accuracy_clean, 4))

**Exercise 2.1:** Is this baseline accuracy higher or lower than the one you computed in Part 1 on the full dataset? Why might dropping passengers with missing ages change the baseline?

*Your answer here.*

### Depth 1: One question

> **Reminder — the decision tree pattern from WS 10.2:**
>
> ```python
> my_model = DecisionTreeClassifier(max_depth=___)
> my_model.fit(X, y)
>
> plt.figure(figsize=(10, 4))
> plot_tree(my_model, feature_names=features, class_names=[_____, _____], filled=True)
> plt.title('_____')
> plt.show()
> ```
>
> Fill in `max_depth`, `class_names` (the label names for your data — here that's `'did not survive'` and `'survived'`), and a descriptive `title`. The variable names `my_model`, `X`, `y`, and `features` are up to you — just be consistent when you reuse them.
>
> After training, use `my_model.predict(X)` to get predictions and `pd.crosstab(y, predictions, rownames=['actual'], colnames=['predicted'])` for the confusion matrix.

**Exercise 2.2:** Train a `DecisionTreeClassifier` with `max_depth=1` on `X` and `y`. Plot the tree. What single question does the tree ask?

In [ ]:
# Your code here

*Your answer here.*

**Exercise 2.3:** Compute predictions from your depth-1 model and build the confusion matrix with `pd.crosstab()`.

In [ ]:
# Your code here

**Exercise 2.4:** Write the four values from the confusion matrix:

TP = ___  &nbsp;&nbsp;  FP = ___  &nbsp;&nbsp;  FN = ___  &nbsp;&nbsp;  TN = ___

Compare this to the "always predict did not survive" confusion matrix from Exercise 1.6. What improved? What got worse?

*Your answer here.*

### Depth 2: Two questions

**Exercise 2.5:** Train a new tree with `max_depth=2`. Plot it (use a wider figure — `figsize=(12, 5)`). Describe the rule in plain English: who does this model predict survived?

In [ ]:
# Your code here

*Your answer here.*

**Exercise 2.6:** Compute predictions and build the confusion matrix for the depth-2 tree.

In [ ]:
# Your code here

Write the four values:

TP = ___  &nbsp;&nbsp;  FP = ___  &nbsp;&nbsp;  FN = ___  &nbsp;&nbsp;  TN = ___

How did the confusion matrix change compared to depth 1?

*Your answer here.*

### Depth 4

**Exercise 2.7:** Train and plot a depth-4 tree (use `figsize=(20, 8)` — it will be big). Compute predictions and build the confusion matrix.

In [ ]:
# Your code here

Write the four values:

TP = ___  &nbsp;&nbsp;  FP = ___  &nbsp;&nbsp;  FN = ___  &nbsp;&nbsp;  TN = ___

*Your answer here.*

### Comparing all four models

**Exercise 2.8:** You now have four confusion matrices — from always-predict-zero, depth 1, depth 2, and depth 4. Fill in this summary table:

| Model | TP | FP | FN | TN | Accuracy |
|---|---|---|---|---|---|
| Always predict 0 | | | | | |
| Depth 1 | | | | | |
| Depth 2 | | | | | |
| Depth 4 | | | | | |

Which model has the highest accuracy? As the trees got deeper, what happened to the number of false negatives? What happened to the number of false positives?

*Your answer here.*

---

## Part 3: Precision and Recall

You've been tracking TP, FP, FN, and TN across four models. The confusion matrix has all the information — but it's hard to compare models by staring at four numbers each time. We need a way to turn the confusion matrix into a single number that captures a specific kind of quality.

There are two natural questions to ask about a classifier's positive predictions:

**Question 1: When the model says "survived," how often is it right?**

This is called **precision**:

$$\text{Precision} = \frac{TP}{TP + FP} = \frac{TP}{\text{everyone the model predicted survived}}$$

The denominator is everyone the model *predicted* as positive (survived) — both the ones it got right (TP) and the ones it got wrong (FP). So precision measures: **of all positive predictions, what fraction were correct?**

A model with high precision rarely gives false alarms — when it says "yes," you can trust it.

**Question 2: Of everyone who actually survived, how many did the model find?**

This is called **recall**:

$$\text{Recall} = \frac{TP}{TP + FN} = \frac{TP}{\text{everyone who actually survived}}$$

The denominator is everyone who *actually* was positive (survived) — both those the model found (TP) and those it missed (FN). So recall measures: **of all actual positives, what fraction did the model catch?**

Think of it this way: precision looks at the model's "yes" pile and asks "how clean is this pile?" Recall looks at all the real positives and asks "how many did the model find?"

A model with high recall rarely misses real cases — it finds most of the people it should find.

**Exercise 3.1:** Using the TP, FP, FN, and TN values from your depth-1 confusion matrix (Exercise 2.4), compute precision and recall by hand. Show your work.

*Hint:* TP = 197, FP = 64, FN = 93, TN = 360.

*Your answer here.*

**Exercise 3.2:** What about the "always predict 0" model? It has TP = 0 and FP = 0. Try plugging those into the precision formula. What happens? What is its recall? Does the question precision is asking even make sense for this model?

*Your answer here.*

**Exercise 3.3:** The "always predict 0" model had decent accuracy (around 60%). But its recall is 0. What does this tell you about using accuracy alone to judge a model?

*Your answer here.*

---

## Part 4: Writing a Function

You calculated precision and recall by hand in Exercise 3.1. It worked — but you built four models in Part 2, and computing three metrics for each one means repeating the same arithmetic again and again. When you find yourself doing the same steps over and over, that's a sign to teach Python to do it for you.

### Defining a function with `def`

In Python, you can package a block of code and give it a name so you can reuse it. This is called a **function**. Here's the pattern:

```python
def greet(name):
    print("Hello,", name)
```

- `def` tells Python you're defining a function
- `greet` is the name you chose
- `name` is a **parameter** — a placeholder for whatever value you pass in later
- The indented lines are the **body** — the code that runs each time you call the function

After defining a function, you **call** it by writing its name with a value in parentheses:

```python
greet("Mai")    # prints: Hello, Mai
greet("Thanh")  # prints: Hello, Thanh
```

The value you pass in (`"Mai"`) fills in for the parameter (`name`) everywhere it appears in the body. A function can take multiple parameters — just separate them with commas.

**Exercise 4.1:** The cell below defines an `evaluate` function that takes the four confusion matrix values and prints precision, recall, and accuracy. Replace each `___` with the correct formula.

> **Reminder:**
> - Precision = TP / (TP + FP)
> - Recall = TP / (TP + FN)
> - Accuracy = (TP + TN) / (TP + FP + FN + TN)

In [ ]:
def evaluate(tp, fp, fn, tn):
    precision = ___
    recall = ___
    accuracy = ___
    print("Precision:", round(precision, 3))
    print("Recall:   ", round(recall, 3))
    print("Accuracy: ", round(accuracy, 3))

Test your function on the depth-1 model. The output should match your hand calculation from Exercise 3.1.

In [ ]:
evaluate(197, 64, 93, 360)

**Exercise 4.2:** Now use `evaluate()` to compute metrics for the depth-2 and depth-4 models. Use the TP, FP, FN, TN values from your confusion matrices in Part 2.

In [ ]:
# Your code here

Fill in the comparison table. (The always-predict-0 row is filled in from Exercise 3.2 — `evaluate()` can't handle it because precision would be 0/0.)

| Model | Precision | Recall | Accuracy |
|---|---|---|---|
| Always predict 0 | undefined (0/0) | 0.000 | 0.594 |
| Depth 1 | | | |
| Depth 2 | | | |
| Depth 4 | | | |

*Your answer here.*

**Exercise 4.3:** As the trees got deeper, what happened to precision? What happened to recall? Did they move in the same direction or opposite directions?

*Your answer here.*

---

## Part 5: When the Stakes Are Real

The Titanic is a useful example for learning the mechanics of precision and recall, but the passengers are long gone — there is no decision to make. Let's move to a setting where classifier errors affect real people's lives right now.

Banks use models to predict whether someone will **default** (fail to pay back) on a loan. If the model predicts default, the bank may reject the application. If the model predicts no default, the bank may approve it.

Run the cell below to load a dataset of 30,000 credit card customers from a bank in Taiwan ([source: UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients)). The column `default` is 1 if the customer failed to pay the next month's bill and 0 if they paid.

In [ ]:
credit = pd.read_csv("https://raw.githubusercontent.com/statisfactions/QRAI-materials/main/data/taiwan_credit.csv")
credit.head(10)

**Exercise 5.1:** How many customers are in the dataset? How many defaulted? What fraction of customers defaulted?

In [ ]:
# Your code here

*Your answer here.*

**Exercise 5.2:** Based on that fraction, what accuracy would an always-predict-zero model get? (Think first, then run the cell below to check.)

*Your answer here.*

In [ ]:
predictions_no_default = np.zeros(len(credit))

accuracy_baseline = (predictions_no_default == credit["default"]).sum() / len(credit)
print("Baseline accuracy:", round(accuracy_baseline, 4))

pd.crosstab(credit["default"], predictions_no_default, rownames=['actual'], colnames=['predicted'])

Same pattern as the Titanic: TP = 0, FP = 0, FN = 6,636, TN = 23,364. The model has 77.9% accuracy but misses every single defaulter — recall is zero.

Now let's build a real model. We'll use four features from the dataset:

- `LIMIT_BAL` — the customer's credit limit
- `AGE` — the customer's age
- `PAY_AMT1` — how much the customer paid last month
- `BILL_AMT1` — how much the customer owed last month

In [ ]:
features_credit = ["LIMIT_BAL", "AGE", "PAY_AMT1", "BILL_AMT1"]
X_credit = credit[features_credit]
y_credit = credit["default"]

**Exercise 5.3:** Train a decision tree with `max_depth=4` on the credit data. Compute predictions and build the confusion matrix.

In [ ]:
# Your code here

**Exercise 5.4:** Use your `evaluate()` function on the confusion matrix values from the depth-4 tree. How does this model compare to the always-predict-zero baseline?

In [ ]:
# Your code here

*Your answer here.*

**Exercise 5.5:** Think about what each type of error means for a real person:

- A **false positive** means the model says someone will default, but they would have paid. What might the bank do to this person?
- A **false negative** means the model says someone will pay, but they actually default. What is the cost to the bank?

In this setting, which concerns you more — low precision or low recall? There is no single right answer — explain your reasoning.

*Your answer here.*

**Exercise 5.6:** Imagine you are advising the bank. They show you this model and say: "It has 78% accuracy — should we use it?" What would you tell them? What information would you want to know before deciding?

*Your answer here.*

---

*Worksheet created by Ethan C. Brown in collaboration with Claude Code.*